# Validation

One of the main value props of Pydantic is that it provides a wide variety of validation mechanisms. A lot of validation can be controlled by using the right type for the field. In case the validation logic cannot be represented with a type, I can add my own validation by -
  * Using Pydantic's [custom types](https://docs.pydantic.dev/latest/api/types/#pydantic.types)
  * Setting field constriaints using `Field()`
  * Writing custom validation functions

In [35]:
from __future__ import annotations
from pydantic import *  # type: ignore
from datetime import datetime
from pprint import pprint
from enum import StrEnum, auto
from typing import Any, cast, Self

## Custom Types
These are pretty straighforward to use. Just annotate the type with this custom type. Pydantic has a long list of interesting custom types that should meet most of my requirements. Note, not all custom types are about validation. Different types support different use cases, validation is supported by some of them.

In [2]:
class User(BaseModel):
    id: int
    username: str
    created_at: PastDatetime

In [3]:
User(id=1, username="APTG", created_at=datetime(1980, 1, 1))

User(id=1, username='APTG', created_at=datetime.datetime(1980, 1, 1, 0, 0))

In [4]:
try:
    User(id=2, username="APTG", created_at=datetime(2050, 1, 1))
except ValidationError as ve:
    pprint(ve.errors())

[{'input': datetime.datetime(2050, 1, 1, 0, 0),
  'loc': ('created_at',),
  'msg': 'Input should be in the past',
  'type': 'datetime_past',
  'url': 'https://errors.pydantic.dev/2.11/v/datetime_past'}]


## Field Constraints

I can add a bunch of constraints to individual fields that will be followed during validaiton.

  * Numeric Constraints
    - `gt` and `lt`
    - `ge` and `le`
    - `multiple_of`
    - `allow_inf_nan`

  * Iterable/String Constraints
    - `min_length`
    - `max_length`
    - `pattern`

  * Decimal Constraints
    - `max_digits`
    - `decimal_places`  

In [5]:
class FunWithNumbers(BaseModel):
    semi_positive: int = Field(ge=0)
    logscale: int = Field(multiple_of=10)
    unstable_num: float = Field(allow_inf_nan=True)

In [6]:
FunWithNumbers(semi_positive=0, logscale=100, unstable_num=float("-inf"))

FunWithNumbers(semi_positive=0, logscale=100, unstable_num=-inf)

In [7]:
try:
    FunWithNumbers(semi_positive=-1, logscale=3, unstable_num=1.1)
except ValidationError as ve:
    pprint(ve.errors())

[{'ctx': {'ge': 0},
  'input': -1,
  'loc': ('semi_positive',),
  'msg': 'Input should be greater than or equal to 0',
  'type': 'greater_than_equal',
  'url': 'https://errors.pydantic.dev/2.11/v/greater_than_equal'},
 {'ctx': {'multiple_of': 10},
  'input': 3,
  'loc': ('logscale',),
  'msg': 'Input should be a multiple of 10',
  'type': 'multiple_of',
  'url': 'https://errors.pydantic.dev/2.11/v/multiple_of'}]


## Validation Functions

Ref: https://docs.pydantic.dev/latest/concepts/validators/

These can be at a per-field level -- using the `@field_validator()`, or at the model level -- using the `@model_validator()`. In either case there are four different types of custom validators depending on when in the validation lifecycle they run -
  * Before validators - run before the Pydantic validators run.
  * After validators - run after the Pydantic validators run.
  * Plain validators - run instead of Pydantic validators, i.e., the Pydantic validators never run.
  * Wrap validators - wrap Pydantic validators, and they can choose if and when to call the Pydantic validators.

The per-field validators are run as each field is processed, but before the full object is instantiated, this means that these functions need to be class methods. The model validators on the other hand can be instance methods if the validator is supposed to be run after. In all other cases they still have to be class methods.  

If the custom function wants to raise an error, the simplest thing to do is to raise the plain old `ValueError`. Pydantic will convert this into the usual `ValidationError`. However, I do have the option to raise a couple of other errors like `AssertionError` or `PydanticCustomError` for advanced scenarios.

The validator functions can also optionally get additional validation data and validation context, but this too seems like advanced scenarios that I don't want to get into unless I actually need to.

Pydnantic also has a cool feature where I can decorate any old Python function with type annotations with `@validate_call` and get the same benefits of a model valiator. For now, I am not going to use while I get more familiar with Python's own type annotations.

### Before Validators

Before-validators are given the user input directly, which means the input can be anything. The validator needs to be robust to any and all sorts of input. As can be seen in the model before-validator, it has to deal with a bunch of different edge cases.

In [ ]:
existing_users = ["happy", "cookie", "orange"]


class User(BaseModel):
    id: int
    name: str

    # The input can be Any-thing
    # It seems to work even if @classmethod is missing, but if it is there it must be after!
    @field_validator("name", mode="before")
    @classmethod
    def is_unique(cls, value: Any) -> str:
        if isinstance(value, str):
            if value not in existing_users:
                return value
            else:
                raise ValueError("User name already exists!")

        else:
            raise ValueError("User name must be a string!")

In [9]:
User(id=1, name="APTG")

type=<class 'str'>, APTG


User(id=1, name='APTG')

In [10]:
try:
    User(id=2, name="cookie")
except ValidationError as ve:
    pprint(ve.errors())

type=<class 'str'>, cookie
[{'ctx': {'error': ValueError('User name already exists!')},
  'input': 'cookie',
  'loc': ('name',),
  'msg': 'Value error, User name already exists!',
  'type': 'value_error',
  'url': 'https://errors.pydantic.dev/2.11/v/value_error'}]


In [11]:
try:
    User(id=3, name={"username": "Anon"})  # type: ignore
except ValidationError as ve:
    pprint(ve.errors())

type=<class 'dict'>, {'username': 'Anon'}
[{'ctx': {'error': ValueError('User name must be a string!')},
  'input': {'username': 'Anon'},
  'loc': ('name',),
  'msg': 'Value error, User name must be a string!',
  'type': 'value_error',
  'url': 'https://errors.pydantic.dev/2.11/v/value_error'}]


In [43]:
dead_links = ["http://dead1.com", "http://dead2.com"]


class Bookmark(BaseModel):
    id: int
    url: str
    added_on: datetime

    @model_validator(mode="before")
    @classmethod
    def sanitize(cls, data: Any) -> Any:
        if not isinstance(data, dict):
            return data

        # Try to reformat input datestring to ISO
        # If I cannot, leave it for Pydantic to handle
        if "added_on" in data and isinstance(data["added_on"], str):
            try:
                ds = datetime.strptime(
                    data["added_on"], "%a, %b %d, %Y | %I:%M %p"
                ).isoformat()
                data["added_on"] = ds
            except ValueError as ve:
                pass

        if "url" in data and isinstance(data["url"], str):
            if data["url"] in dead_links:
                raise ValueError(f"{data['url']} is not active!")

        return data

In [44]:
Bookmark(id=1, url="http://happy.com", added_on="Mon, Dec 01, 1980 | 03:33 PM")  # type: ignore

Bookmark(id=1, url='http://happy.com', added_on=datetime.datetime(1980, 12, 1, 15, 33))

In [45]:
try:
    Bookmark(id=1, url="http://dead1.com", added_on="Mon, Dec 01, 1980 | 03:33 PM")  # type: ignore
except ValidationError as ve:
    pprint(ve.errors())

[{'ctx': {'error': ValueError('http://dead1.com is not active!')},
  'input': {'added_on': '1980-12-01T15:33:00',
            'id': 1,
            'url': 'http://dead1.com'},
  'loc': (),
  'msg': 'Value error, http://dead1.com is not active!',
  'type': 'value_error',
  'url': 'https://errors.pydantic.dev/2.11/v/value_error'}]


### After Validators
After validators are guaranteed to get the actual declared type of the field. If it is a model validator, it can be an instance method as opposed to a class method. For instance methods, the return type is `Self` which is Python's shorthand for the generic self.

```python
class MyClass:
    def func[T: MyClass](self: T) -> T:
        ...
        return self
```

Can be re-written as

```python
class MyClass:
    def func(self) -> Self:
        ...
        return self
```

In [34]:
class Counter(BaseModel):
    value: int

    @model_validator(mode="after")
    def power_of_two(self) -> Self:
        val = self.value
        val = val if val > 0 and (val & (val - 1)) == 0 else 1
        self.value = val
        return self

In [36]:
Counter(value=3)

Counter(value=1)

In [37]:
Counter(value=8)

Counter(value=8)

In [ ]:
# Because the validator runs after the object has been initialized, the method gets the parsed int
Counter(value="008")  # type: ignore

Counter(value=8)

### Wrap Validators

These functions get the appropriate Pydantic validation function as part of their input. They can decied when/if to run the Pydantic validation.

In [47]:
class Name(BaseModel):
    value: str = Field(max_length=5)

    @field_validator("value", mode="wrap")
    @classmethod
    def truncate(cls, value: Any, handler: ValidatorFunctionWrapHandler) -> str:
        try:
            return handler(value)
        except ValidationError as err:
            if err.errors()[0]["type"] == "string_too_long":
                return handler(value[:5])
            else:
                raise

In [48]:
Name(value="abcdefgh")

Name(value='abcde')